In [ ]:
#update method in BallTracker takes detection results, converts bounding boxes to center points, and stores them in a buffer. It then calculates the average ball position (centroid) and selects the ball closest to it, assuming there's only one ball on the field and its movement is physically realistic.     

import supervision as sv
import numpy as np
from ultralytics import YOLO
from collections import deque
from scenedetect.backends import VideoCaptureAdapter
from pathlib import Path

class BallTracker:
    def __init__(self, buffer_size: int = 10): # 
        self.positions = deque(maxlen=buffer_size)
        self.speed_history = deque(maxlen=20)

    def update(self, detections: sv.Detections, frame_number: int=None) -> sv.Detections:
        has_detection = len(detections)>0
        
        if has_detection:
            xy = detections.get_anchors_coordinates(sv.Position.CENTER)
            print(f"xy is: {xy}")
            self.positions.append(xy)
            print(f" The buffer state is: {self.buffer}")
        else:
            return detections

        centroid = np.mean(np.concatenate(self.buffer), axis=0)
        print(f"The centroid is: {centroid}")
        distances = np.linalg.norm(xy - centroid, axis=1)
        print(f"The distances are: {distances}")
        index = np.argmin(distances)
        print(f"The index are: {index}")
        return detections[[index]]

from scenedetect import (SceneManager, AdaptiveDetector, ContentDetector, StatsManager, open_video, save_images, split_video_ffmpeg)

values = {"frame_change_vals": [], "scene_id": None, "stats": None}
def scene_change(VIDEO_PATH, detector = ContentDetector()): #AdaptiveDetector), ThresholdDetector, 
    video = open_video(VIDEO_PATH) 
    scene_id_array = np.full(video.duration.get_frames(), 0, dtype=int) # Initialise all frame to scene 0   
    stats_manager= StatsManager()
    scene_manager = SceneManager(stats_manager=stats_manager)
    scene_manager.add_detector(detector) 
    def scene_callback(frame_im, frame_num):
        values["frame_change_vals"].append(frame_num)
        # print(f"Scene cut detected at frame {frame_num}")
    scene_manager.detect_scenes(video=video,duration=video.duration, callback=scene_callback) #show_progress=True
    scene_list = scene_manager.get_scene_list()
    for i, (start_time, end_time) in enumerate(scene_list):
        start_frame = int(start_time.get_frames())
        end_frame = min(int(end_time.get_frames()), len(scene_id_array)) # Min is put to address edge case when it goes outside frame
        scene_id_array[start_frame: end_frame] = i
    values.update({"scene_id": scene_id_array}) #updateing scene_id
    try:
        values["stats"] = stats_manager._frame_metrics # Stores all the metrics in dictionary format along with frame_number
    except Exception as e:
        print("Stats doesn't exist")
    print(f"Lenght of scene list is {len(scene_list)}")
    
    DEBUG = False
    if DEBUG==True:
        pass
        # scene_manager.stats_manager.save_to_csv(csv_file='') # To save csv and find out optimal threshold by visualisations
        # save_images(scene_list, open_video(video_path), image_name_template='$VIDEO_NAME-Scene-$SCENE_NUMBER-$IMAGE_NUMBER', output_dir='./save_images') # Error with this: image_extension='.jpg'
        # visualize_detector_metrics(VIDEO_PATH, output_dir)


VIDEO_PATH = '/Users/spectatr/Downloads/nsl_match2/683efcc10388617b7ff4b9cf/input_001a21f5-580a-4280-98f2-49d4e3fbd28d1748958688 copy.mp4'
OUTPUT_VIDEO_PATH = '/Users/spectatr/Downloads/testing_short1.mp4'
MODEL_PATH = '/Users/spectatr/Downloads/best_datav10_rectified_nano_1920_e261_final.pt'
MODEL_PERSON_PATH = ""
model_first_football = YOLO(MODEL_PATH)
# model_second_person = YOLO(MODEL_PERSON_PATH)
model_second_person = YOLO('yolov8n.pt')
tracker = BallTracker(buffer_size = 7)
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
width, height = video_info.resolution_wh
fps, total_frames = video_info.fps, video_info.total_frames
print(f"FPS is: {fps} and total Frames: {total_frames}")
percentage_to_exclude_from_sides = 0.01 #(1%) percentage amount to remove ball detections.
border_x, border_y = width*percentage_to_exclude_from_sides, height*percentage_to_exclude_from_sides 

all_detections = []
states = {"gap_counter": 0, "total_detections": 0}
buffer_states = {"position_history":deque(maxlen=10), "frame_history": deque(maxlen=10), "confidence_history": deque(maxlen=5), "bbox_history": deque(maxlen=5)}

print(f"----------Running scene_change function--------")
scene_change(VIDEO_PATH)
print(f"--------Ended scene change function --------")

def callback(frame, index):
    # rgb_frame = frame[:, :, ::-1].copy()
    result = model_first_football(frame)[0]
    # result_player = model_second_person(frame)[0]
    # detections_player = sv.Detections.from_ultralytics(result_player)
    detections = sv.Detections.from_ultralytics(result)
    # detections = detections[(detections.class_id==32)] #Uncomment when the model is pretrained or default with many classes
    detections = detections[(detections.xyxy[:, 0] > border_x) & (detections.xyxy[:,1]>border_y) & (detections.xyxy[:,2] < width - border_x) & (detections.xyxy[:,3] < height - border_y)] # to filter out outside frames
    detections = sv.ByteTrack().update_with_detections(detections)
    # detections = sv.DetectionsSmoother().update_with_detections(detections) 
    # Scene detect when Pysene detect
    # scene_id = int(values["scene_id"][index]) if index < len(values["scene_id"]) else -1 # Just handling errors if occured
    # detections.metadata.update({"frame_change": scene_id})
    # Scene change when using camera_view
    import camera_view
    frame_changes_csv = camera_view.camera_view(VIDEO_PATH, Path.cwd()/'camera_view_files')
    


    detections.metadata.update({"frame_number": index}) #Save new metadata here
    detections.metadata.update({"frame_brightness": np.mean, "frame_contrast": np.std(frame)})
    if len(detections)==0: #If no detections are found in a frame
        states["gap_counter"]+=1 # Better way to maintain state to default value of 0 when there are continous detections
        # detections.metadata.update({"gap": states["gap_counter"]}) #Written to only use gap when there are gaps and when no gaps then empty
    else:
        states["gap_counter"]=0
        get_center = detections.get_anchors_coordinates(sv.Position.CENTER) # Format --> [[     666.54      497.88]] <class 'numpy.ndarray'> 2 (1, 2)
        detections.metadata.update({"center_position": get_center})
        #Spatial context metadata.
        detections.metadata.update({"ball_distance_from_center": np.linalg.norm(get_center[0] - np.array([width/2, height/2])), 
                                   "relative_position_x": get_center[0][0]/width,
                                   "relative_position_y": get_center[0][1]/height})
        
        # Additional
        states["total_detections"]+=1

    detections.metadata.update({"gap": states["gap_counter"]}) # Common for both detections state so bring it outside to avoid duplicacy in lines.
    
    sort_keep_indices = np.argsort(detections.confidence)[::-1][:1] # To sort out the highest confidence score box
    detections = detections[sort_keep_indices] #Filter the single box
    
    print(detections.class_id, detections.tracker_id, detections.metadata) # To understand the detections working fine or not
    # detections = tracker.update(detections)
    
    labels = [f"{class_id} -- {round(confidence,2)}" for class_id, confidence in zip(detections.class_id, detections.confidence)]
    annotated_frame = frame.copy()
    annotated_frame = sv.BoxAnnotator().annotate(annotated_frame, detections) 
    annotated_frame = sv.LabelAnnotator().annotate(annotated_frame, detections, labels=labels)
    annotated_frame = sv.TraceAnnotator().annotate(annotated_frame, detections)
    all_detections.append(detections)
    return annotated_frame
sv.process_video(source_path=VIDEO_PATH, target_path=OUTPUT_VIDEO_PATH, callback=callback)

print(f"""Detection rate is: {states["total_detections"]/total_frames*100} \\
      Average speed is: {10} \\
      Max speed is : {12} \\
      Average confidence is: {12} \\
      Longest gap is: max([d.metadata.get("gap_count", 0) for d in gap_frames], default=0)""")

# Line code testing
# start = sv.Point(400, 800) #To draw point on the annoatated image
# end = sv.Point(500, 900) 
# line_zone = sv.LineZone(start, end) # To draw line on the annotated image
# line_zone_annotator = sv.LineZoneAnnotator(thickness=5, display_text_box=False, display_in_count=False, display_out_count=False)
# annotated_frame = line_zone_annotator.annotate(frame, line_counter=line_zone)
# sv.plot_image(annotated_frame)
# video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
# frame_generator = sv.get_video_frames_generator(VIDEO_PATH)
# frame = next(iter(frame_generator))
# # sv.plot_image(frame, size=(8,8))

FPS is: 60 and total Frames: 643

0: 1088x1920 1 Ball, 305.5ms
Speed: 8.4ms preprocess, 305.5ms inference, 1.3ms postprocess per image at shape (1, 3, 1088, 1920)
[0] [1] {'frame_number': 0, 'frame_brightness': <function mean at 0x11207feb0>, 'frame_contrast': np.float64(50.90899145465872), 'center_position': array([[     666.54,      497.88]], dtype=float32), 'ball_distance_from_center': np.float64(296.4645379145185), 'relative_position_x': np.float32(0.34715804), 'relative_position_y': np.float32(0.46099576)}

0: 1088x1920 1 Ball, 226.5ms
Speed: 6.4ms preprocess, 226.5ms inference, 3.1ms postprocess per image at shape (1, 3, 1088, 1920)
[0] [1] {'frame_number': 1, 'frame_brightness': <function mean at 0x11207feb0>, 'frame_contrast': np.float64(50.93198927123435), 'center_position': array([[     676.53,       499.1]], dtype=float32), 'ball_distance_from_center': np.float64(286.4016912823834), 'relative_position_x': np.float32(0.35236117), 'relative_position_y': np.float32(0.46213153)}

KeyboardInterrupt: 

In [20]:
import sys
if 'camera_view' in sys.modules: #This is done for ipynb notebooks cached the module once loaded, so changes made in file are not loaded. 
    del sys.modules['camera_view']
import camera_view
import time, os
from pathlib import Path
video_path = '/Users/spectatr/Downloads/VAR/VAR_12.mp4'
start = time.time()
frame_changes_csv = camera_view.camera_view(video_path, Path.cwd()/'camera_view_files2')
frame_changes_csv


Comparing frames 267 and 268: Similarity score = 20 -> Not Similar
Comparing frames 267 and 268: Similarity score = 20 -> Not Similar
Comparing frames 603 and 604: Similarity score = 28 -> Not Similar
Comparing frames 603 and 604: Similarity score = 28 -> Not Similar
Comparing frames 1013 and 1014: Similarity score = 30 -> Not Similar
Comparing frames 1013 and 1014: Similarity score = 30 -> Not Similar
Comparing frames 1229 and 1230: Similarity score = 32 -> Not Similar
Comparing frames 1229 and 1230: Similarity score = 32 -> Not Similar
Comparing frames 1944 and 1945: Similarity score = 20 -> Not Similar
Comparing frames 1944 and 1945: Similarity score = 20 -> Not Similar
Comparing frames 2249 and 2250: Similarity score = 30 -> Not Similar
Comparing frames 2249 and 2250: Similarity score = 30 -> Not Similar
Comparing frames 2571 and 2572: Similarity score = 30 -> Not Similar
Comparing frames 2571 and 2572: Similarity score = 30 -> Not Similar
Comparing frames 3020 and 3021: Similarity

'/Users/spectatr/Downloads/general_scripts/model_improvement_experiments/camera_view_files2/result2_frame_change.csv'

In [ ]:
import pandas as pd
df = pd.read_csv('')

In [36]:
class ImprovedBallTracker:
    def __init__(self, 
                 buffer_size: int = 10, 
                 max_distance_threshold: float = 100.0,
                 confidence_threshold: float = 0.3,
                 ball_class_id: int = 32,
                 max_velocity_threshold: float = 50.0):
        """
        Enhanced ball tracker with outlier removal capabilities
        
        Args:
            buffer_size: Number of recent positions to keep in memory
            max_distance_threshold: Maximum allowed distance from centroid (pixels)
            confidence_threshold: Minimum confidence score for detections
            ball_class_id: Class ID for ball in your model (32 for sports ball in COCO)
            max_velocity_threshold: Maximum allowed velocity between frames (pixels/frame)
        """
        self.buffer = deque(maxlen=buffer_size)
        self.velocity_buffer = deque(maxlen=buffer_size-1)
        self.max_distance_threshold = max_distance_threshold
        self.confidence_threshold = confidence_threshold
        self.max_velocity_threshold = max_velocity_threshold
        self.last_position = None
        self.frame_count = 0

    
    def _calculate_velocity(self, current_pos: np.ndarray) -> float:
        """Calculate velocity between current and last position"""
        if self.last_position is None:
            return 0.0
        return np.linalg.norm(current_pos - self.last_position)
    
    def _is_physically_plausible(self, position: np.ndarray) -> bool:
        """Check if the position change is physically plausible"""
        if self.last_position is None:
            return True
            
        velocity = self._calculate_velocity(position)
        return velocity <= self.max_velocity_threshold
    
    def _get_weighted_centroid(self) -> np.ndarray:
        """Calculate weighted centroid giving more importance to recent positions"""
        if len(self.buffer) == 0:
            return None
            
        positions = np.concatenate(self.buffer)
        if len(positions) == 0:
            return None
            
        # Apply exponential weighting (recent positions have higher weight)
        weights = np.exp(np.linspace(-1, 0, len(self.buffer)))
        weights = np.repeat(weights, [len(pos) for pos in self.buffer])
        weights = weights / np.sum(weights)
        
        # Calculate weighted centroid
        if len(weights) == len(positions):
            weighted_centroid = np.average(positions, axis=0, weights=weights)
        else:
            # Fallback to simple mean if shapes don't match
            weighted_centroid = np.mean(positions, axis=0)
            
        return weighted_centroid
    
    def _remove_outliers_by_distance(self, detections: sv.Detections) -> sv.Detections:
        """Remove detections that are too far from the expected position"""
        if len(detections) == 0 or len(self.buffer) < 2:
            return detections
            
        xy = detections.get_anchors_coordinates(sv.Position.CENTER)
        centroid = self._get_weighted_centroid()
        
        if centroid is None:
            return detections
            
        # Calculate distances from weighted centroid
        distances = np.linalg.norm(xy - centroid, axis=1)
        
        # Filter out detections that are too far
        valid_mask = distances <= self.max_distance_threshold
        
        return detections[valid_mask]
    
    def _remove_outliers_by_velocity(self, detections: sv.Detections) -> sv.Detections:
        """Remove detections with implausible velocity changes"""
        if len(detections) == 0 or self.last_position is None:
            return detections
            
        xy = detections.get_anchors_coordinates(sv.Position.CENTER)
        
        # Check velocity for each detection
        valid_indices = []
        for i, pos in enumerate(xy):
            if self._is_physically_plausible(pos):
                valid_indices.append(i)
        
        if len(valid_indices) == 0:
            return sv.Detections.empty()
            
        return detections[np.array(valid_indices)]
    
    def _select_best_detection(self, detections: sv.Detections) -> sv.Detections:
        """Select the best detection based on confidence and position consistency"""
        if len(detections) == 0:
            return detections
        elif len(detections) == 1:
            return detections
        
        xy = detections.get_anchors_coordinates(sv.Position.CENTER)
        
        # If we have historical data, prefer detections closer to expected position
        if len(self.buffer) >= 2:
            centroid = self._get_weighted_centroid()
            if centroid is not None:
                distances = np.linalg.norm(xy - centroid, axis=1)
                # Combine confidence and distance (lower distance is better)
                scores = detections.confidence - (distances / self.max_distance_threshold)
                best_idx = np.argmax(scores)
                return detections[[best_idx]]
        
        # Fallback: select highest confidence detection
        best_idx = np.argmax(detections.confidence)
        return detections[[best_idx]]
    
    def update(self, detections: sv.Detections) -> sv.Detections:
        """
        Update tracker with new detections and return filtered result
        
        Args:
            detections: All detections from the frame
            
        Returns:
            Filtered detections containing only the best ball detection
        """
        self.frame_count += 1
        
        # Step 1: Filter for ball detections with sufficient confidence
        # ball_detections = self._filter_ball_detections(detections)
        
        # if len(ball_detections) == 0:
        #     # No ball detections found
        #     self.buffer.append(np.array([]).reshape(0, 2))
        #     return sv.Detections.empty()
        
        # Step 2: Remove outliers based on distance from expected position
        # filtered_detections = self._remove_outliers_by_distance(ball_detections)
        
        # Step 3: Remove outliers based on velocity constraints
        # filtered_detections = self._remove_outliers_by_velocity(filtered_detections)
        filtered_detections = self._remove_outliers_by_velocity(detections)
        
        
        # Step 4: Select the best remaining detection
        final_detection = self._select_best_detection(filtered_detections)
        
        # Step 5: Update tracking history
        if len(final_detection) > 0:
            xy = final_detection.get_anchors_coordinates(sv.Position.CENTER)
            self.buffer.append(xy)
            
            # Update velocity tracking
            if self.last_position is not None:
                velocity = self._calculate_velocity(xy[0])
                self.velocity_buffer.append(velocity)
            
            self.last_position = xy[0]
        else:
            # No valid detection found
            self.buffer.append(np.array([]).reshape(0, 2))
        
        return final_detection
    
    def get_trajectory(self) -> np.ndarray:
        """Get the current trajectory as a numpy array"""
        valid_positions = [pos for pos in self.buffer if len(pos) > 0]
        if not valid_positions:
            return np.array([]).reshape(0, 2)
        return np.vstack(valid_positions)
    
    def get_average_velocity(self) -> float:
        """Get the average velocity over recent frames"""
        if len(self.velocity_buffer) == 0:
            return 0.0
        return np.mean(list(self.velocity_buffer))

tracker = ImprovedBallTracker(buffer_size=5, max_distance_threshold=80, confidence_threshold=0.3, ball_class_id=0, max_velocity_threshold=40)

In [40]:
import supervision as sv
import numpy as np
from ultralytics import YOLO
from collections import deque
from typing import Optional, List, Tuple, Dict, Any
import cv2

class EnhancedBallTracker:
    def __init__(
        self, 
        buffer_size: int = 10,
        speed_threshold: float = 500.0,  # pixels per frame
        confidence_threshold: float = 0.5,
        max_distance_threshold: float = 200.0,  # max pixels from centroid
        min_detections_for_speed: int = 2
    ):
        self.buffer = deque(maxlen=buffer_size)
        self.position_buffer = deque(maxlen=buffer_size)
        self.frame_numbers = deque(maxlen=buffer_size)
        self.confidence_buffer = deque(maxlen=buffer_size)
        self.speed_buffer = deque(maxlen=buffer_size - 1)
        
        # Thresholds
        self.speed_threshold = speed_threshold
        self.confidence_threshold = confidence_threshold
        self.max_distance_threshold = max_distance_threshold
        self.min_detections_for_speed = min_detections_for_speed
        
        # Frame tracking
        self.current_frame = 0
        
        # Statistics
        self.stats = {
            'total_detections': 0,
            'outliers_removed': 0,
            'speed_outliers': 0,
            'distance_outliers': 0,
            'confidence_outliers': 0
        }

    def calculate_speed(self, pos1: np.ndarray, pos2: np.ndarray, frame_diff: int = 1) -> float:
        """Calculate speed between two positions in pixels per frame"""
        if frame_diff <= 0:
            return 0.0
        distance = np.linalg.norm(pos2 - pos1)
        return distance / frame_diff

    def is_speed_outlier(self, current_pos: np.ndarray) -> bool:
        """Check if current position creates unrealistic speed"""
        if len(self.position_buffer) < self.min_detections_for_speed:
            return False
        
        prev_pos = self.position_buffer[-1]
        prev_frame = self.frame_numbers[-1]
        frame_diff = self.current_frame - prev_frame
        
        if frame_diff <= 0:
            return False
            
        speed = self.calculate_speed(prev_pos, current_pos, frame_diff)
        return speed > self.speed_threshold

    def is_distance_outlier(self, current_pos: np.ndarray) -> bool:
        """Check if current position is too far from trajectory centroid"""
        if len(self.position_buffer) < 2:
            return False
        
        # Calculate centroid of recent positions
        recent_positions = np.array(list(self.position_buffer))
        centroid = np.mean(recent_positions, axis=0)
        distance = np.linalg.norm(current_pos - centroid)
        
        return distance > self.max_distance_threshold

    def filter_detections(self, detections: sv.Detections) -> sv.Detections:
        """Filter detections based on confidence and other criteria"""
        if len(detections) == 0:
            return detections
        
        # Filter by confidence
        high_conf_mask = detections.confidence >= self.confidence_threshold
        
        if not np.any(high_conf_mask):
            # If no high confidence detections, keep the highest confidence one
            best_idx = np.argmax(detections.confidence)
            return detections[[best_idx]]
        
        return detections[high_conf_mask]

    def select_best_detection(self, detections: sv.Detections) -> Optional[sv.Detections]:
        """Select the best detection from multiple candidates"""
        if len(detections) == 0:
            return None
        
        if len(detections) == 1:
            return detections
        
        xy = detections.get_anchors_coordinates(sv.Position.CENTER)
        
        # If we have trajectory history, use it to select best detection
        if len(self.position_buffer) > 0:
            centroid = np.mean(np.array(list(self.position_buffer)), axis=0)
            distances = np.linalg.norm(xy - centroid, axis=1)
            best_idx = np.argmin(distances)
            return detections[[best_idx]]
        
        # Otherwise, select highest confidence
        best_idx = np.argmax(detections.confidence)
        return detections[[best_idx]]

    def update_buffers(self, detection: sv.Detections):
        """Update all tracking buffers with new detection"""
        if len(detection) == 0:
            return
        
        center = detection.get_anchors_coordinates(sv.Position.CENTER)[0]
        confidence = detection.confidence[0]
        
        # Update buffers
        self.buffer.append(detection.get_anchors_coordinates(sv.Position.CENTER))
        self.position_buffer.append(center)
        self.frame_numbers.append(self.current_frame)
        self.confidence_buffer.append(confidence)
        
        # Calculate and store speed if possible
        if len(self.position_buffer) >= 2:
            prev_pos = list(self.position_buffer)[-2]
            prev_frame = list(self.frame_numbers)[-2]
            frame_diff = self.current_frame - prev_frame
            speed = self.calculate_speed(prev_pos, center, frame_diff)
            self.speed_buffer.append(speed)

    def update(self, detections: sv.Detections, frame_number: Optional[int] = None) -> sv.Detections:
        """Main update method with outlier removal"""
        if frame_number is not None:
            self.current_frame = frame_number
        else:
            self.current_frame += 1
        
        self.stats['total_detections'] += len(detections)
        
        if len(detections) == 0:
            return detections
        
        # Step 1: Filter by confidence
        filtered_detections = self.filter_detections(detections)
        
        # Step 2: Check for outliers
        valid_detections = []
        
        for i in range(len(filtered_detections)):
            single_detection = filtered_detections[[i]]
            center = single_detection.get_anchors_coordinates(sv.Position.CENTER)[0]
            confidence = single_detection.confidence[0]
            
            is_outlier = False
            
            # Check confidence outlier
            if confidence < self.confidence_threshold:
                is_outlier = True
                self.stats['confidence_outliers'] += 1
            
            # Check speed outlier
            elif self.is_speed_outlier(center):
                is_outlier = True
                self.stats['speed_outliers'] += 1
            
            # Check distance outlier
            elif self.is_distance_outlier(center):
                is_outlier = True
                self.stats['distance_outliers'] += 1
            
            if not is_outlier:
                valid_detections.append(i)
        
        # Step 3: Select best detection from valid ones
        if valid_detections:
            valid_detections_obj = filtered_detections[valid_detections]
            best_detection = self.select_best_detection(valid_detections_obj)
        else:
            # If all are outliers, skip this frame or use fallback logic
            self.stats['outliers_removed'] += len(filtered_detections)
            return sv.Detections.empty()
        
        # Step 4: Update buffers
        if best_detection and len(best_detection) > 0:
            self.update_buffers(best_detection)
            return best_detection
        
        return sv.Detections.empty()

    def get_trajectory(self) -> List[Tuple[float, float]]:
        """Get the current trajectory as list of (x, y) coordinates"""
        return [(pos[0], pos[1]) for pos in self.position_buffer]

    def get_speeds(self) -> List[float]:
        """Get the speed history"""
        return list(self.speed_buffer)

    def get_stats(self) -> Dict[str, Any]:
        """Get tracking statistics"""
        stats = self.stats.copy()
        if stats['total_detections'] > 0:
            stats['outlier_percentage'] = (stats['outliers_removed'] / stats['total_detections']) * 100
        else:
            stats['outlier_percentage'] = 0
        return stats

    def reset(self):
        """Reset the tracker"""
        self.buffer.clear()
        self.position_buffer.clear()
        self.frame_numbers.clear()
        self.confidence_buffer.clear()
        self.speed_buffer.clear()
        self.current_frame = 0
        self.stats = {
            'total_detections': 0,
            'outliers_removed': 0,
            'speed_outliers': 0,
            'distance_outliers': 0,
            'confidence_outliers': 0
        }


class VersatileVisualizer:
    """Versatile visualization class for ball tracking results"""
    
    def __init__(
        self,
        show_trajectory: bool = True,
        show_speed: bool = True,
        show_confidence: bool = True,
        trajectory_length: int = 30,
        trajectory_thickness: int = 3
    ):
        self.show_trajectory = show_trajectory
        self.show_speed = show_speed
        self.show_confidence = show_confidence
        self.trajectory_length = trajectory_length
        self.trajectory_thickness = trajectory_thickness
        
        # Annotation components
        self.box_annotator = sv.BoxAnnotator(
            color=sv.Color.GREEN,
            thickness=2
        )
        self.label_annotator = sv.LabelAnnotator(
            color=sv.Color.GREEN,
            text_color=sv.Color.WHITE,
            text_scale=0.7,
            text_thickness=2
        )
        self.trace_annotator = sv.TraceAnnotator(
            color=sv.Color.RED,
            thickness=self.trajectory_thickness,
            trace_length=self.trajectory_length
        )
        
        # Custom colors for different confidence levels
        self.confidence_colors = {
            'high': sv.Color.GREEN,    # >= 0.8
            'medium': sv.Color.YELLOW, # 0.5-0.8
            'low': sv.Color.RED        # < 0.5
        }

    def get_confidence_color(self, confidence: float) -> sv.Color:
        """Get color based on confidence level"""
        if confidence >= 0.8:
            return self.confidence_colors['high']
        elif confidence >= 0.5:
            return self.confidence_colors['medium']
        else:
            return self.confidence_colors['low']

    def create_labels(
        self, 
        detections: sv.Detections, 
        tracker: EnhancedBallTracker,
        frame_number: int
    ) -> List[str]:
        """Create informative labels for detections"""
        labels = []
        
        for i, (class_id, confidence) in enumerate(zip(detections.class_id, detections.confidence)):
            label_parts = []
            
            # Basic info
            label_parts.append(f"Ball: {confidence:.2f}")
            
            # Speed info
            if self.show_speed and len(tracker.get_speeds()) > 0:
                current_speed = tracker.get_speeds()[-1]
                label_parts.append(f"Speed: {current_speed:.1f}")
            
            # Frame number
            label_parts.append(f"Frame: {frame_number}")
            
            labels.append(" | ".join(label_parts))
        
        return labels

    def draw_trajectory_info(
        self, 
        frame: np.ndarray, 
        tracker: EnhancedBallTracker
    ) -> np.ndarray:
        """Draw additional trajectory information on frame"""
        if not self.show_trajectory:
            return frame
        
        trajectory = tracker.get_trajectory()
        if len(trajectory) < 2:
            return frame
        
        # Draw trajectory line
        points = np.array(trajectory, dtype=np.int32)
        
        for i in range(1, len(points)):
            # Fade color based on age
            alpha = i / len(points)
            color = (0, int(255 * alpha), 0)  # Green with fading
            cv2.line(frame, tuple(points[i-1]), tuple(points[i]), color, 2)
        
        # Draw speed graph (mini visualization)
        if self.show_speed and len(tracker.get_speeds()) > 1:
            self.draw_speed_graph(frame, tracker.get_speeds())
        
        return frame

    def draw_speed_graph(self, frame: np.ndarray, speeds: List[float]) -> None:
        """Draw a mini speed graph on the frame"""
        if len(speeds) < 2:
            return
        
        # Position for mini graph (top-right corner)
        graph_w, graph_h = 200, 100
        start_x = frame.shape[1] - graph_w - 20
        start_y = 20
        
        # Background
        cv2.rectangle(frame, (start_x, start_y), (start_x + graph_w, start_y + graph_h), (0, 0, 0), -1)
        cv2.rectangle(frame, (start_x, start_y), (start_x + graph_w, start_y + graph_h), (255, 255, 255), 2)
        
        # Scale speeds to graph height
        max_speed = max(speeds) if speeds else 1
        scaled_speeds = [(speed / max_speed) * (graph_h - 20) for speed in speeds[-20:]]  # Last 20 speeds
        
        # Draw speed line
        points = []
        for i, speed in enumerate(scaled_speeds):
            x = start_x + 10 + int((i / len(scaled_speeds)) * (graph_w - 20))
            y = start_y + graph_h - 10 - int(speed)
            points.append((x, y))
        
        if len(points) > 1:
            for i in range(1, len(points)):
                cv2.line(frame, points[i-1], points[i], (0, 255, 0), 2)
        
        # Add labels
        cv2.putText(frame, f"Speed: {speeds[-1]:.1f}", (start_x + 5, start_y + 15), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

    def annotate_frame(
        self, 
        frame: np.ndarray, 
        detections: sv.Detections, 
        tracker: EnhancedBallTracker,
        frame_number: int
    ) -> np.ndarray:
        """Main annotation method"""
        annotated_frame = frame.copy()
        
        if len(detections) == 0:
            return self.draw_trajectory_info(annotated_frame, tracker)
        
        # Create dynamic box annotator based on confidence
        if self.show_confidence and len(detections.confidence) > 0:
            confidence = detections.confidence[0]
            color = self.get_confidence_color(confidence)
            dynamic_box_annotator = sv.BoxAnnotator(color=color, thickness=2)
            dynamic_label_annotator = sv.LabelAnnotator(color=color, text_color=sv.Color.WHITE)
        else:
            dynamic_box_annotator = self.box_annotator
            dynamic_label_annotator = self.label_annotator
        
        # Annotate with boxes and labels
        annotated_frame = dynamic_box_annotator.annotate(annotated_frame, detections)
        
        labels = self.create_labels(detections, tracker, frame_number)
        annotated_frame = dynamic_label_annotator.annotate(annotated_frame, detections, labels=labels)
        
        # Add trajectory visualization
        annotated_frame = self.draw_trajectory_info(annotated_frame, tracker)
        
        return annotated_frame


# Usage Example
def process_video_with_enhanced_tracking(
    video_path: str,
    output_path: str,
    model_path: str,
    **tracker_kwargs
):
    """Process video with enhanced ball tracking"""
    
    # Initialize components
    model = YOLO(model_path)
    tracker = EnhancedBallTracker(**tracker_kwargs)
    visualizer = VersatileVisualizer()
    
    # Video info
    video_info = sv.VideoInfo.from_video_path(video_path)
    width, height = video_info.resolution_wh
    
    # Border filtering
    percentage_to_exclude_from_sides = 0.01
    border_x = width * percentage_to_exclude_from_sides
    border_y = height * percentage_to_exclude_from_sides
    
    all_detections = []
    frame_number = 0
    
    def callback(frame: np.ndarray) -> np.ndarray:
        nonlocal frame_number
        frame_number += 1
        
        # Run YOLO detection
        result = model(frame)[0]
        detections = sv.Detections.from_ultralytics(result)
        
        # Filter detections (borders, class, etc.)
        detections = detections[
            (detections.xyxy[:, 0] > border_x) & 
            (detections.xyxy[:, 1] > border_y) & 
            (detections.xyxy[:, 2] < width - border_x) & 
            (detections.xyxy[:, 3] < height - border_y)
        ]
        
        # Keep only highest confidence detection initially
        if len(detections) > 1:
            sort_keep_indices = np.argsort(detections.confidence)[::-1][:1]
            detections = detections[sort_keep_indices]
        
        # Update tracker with outlier removal
        tracked_detections = tracker.update(detections, frame_number)
        
        # Visualize
        annotated_frame = visualizer.annotate_frame(
            frame, tracked_detections, tracker, frame_number
        )
        
        all_detections.append(tracked_detections)
        
        return annotated_frame
    
    # Process video
    sv.process_video(
        source_path=video_path,
        target_path=output_path,
        callback=callback
    )
    
    # Print statistics
    stats = tracker.get_stats()
    print("\n=== Tracking Statistics ===")
    for key, value in stats.items():
        print(f"{key}: {value}")
    
    return all_detections, tracker

# Updated usage example replacing your current code

import supervision as sv
import numpy as np
from ultralytics import YOLO

# Your file paths
VIDEO_PATH = '/Users/spectatr/Downloads/nsl_match2/683efcc10388617b7ff4b9cf/input_001a21f5-580a-4280-98f2-49d4e3fbd28d1748958688.mp4'
OUTPUT_VIDEO_PATH = '/Users/spectatr/Downloads/testing3.mp4'
MODEL_PATH = '/Users/spectatr/Downloads/best_datav10_rectified_nano_1920_e261_final.pt'

# Initialize enhanced tracker with custom parameters
tracker = EnhancedBallTracker(
    buffer_size=10,
    speed_threshold=500.0,  # Adjust based on your sport (pixels per frame)
    confidence_threshold=0.3,  # Lower threshold for initial filtering
    max_distance_threshold=200.0,  # Max distance from trajectory centroid
    min_detections_for_speed=2
)

# Initialize versatile visualizer
visualizer = VersatileVisualizer(
    show_trajectory=True,
    show_speed=True,
    show_confidence=True,
    trajectory_length=30
)

# Load model
model = YOLO(MODEL_PATH)

# Video setup
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
width, height = video_info.resolution_wh
percentage_to_exclude_from_sides = 0.01
border_x, border_y = width * percentage_to_exclude_from_sides, height * percentage_to_exclude_from_sides

all_detections = []
frame_number = 0
# Alternative: Use the convenience function
all_detections, tracker = process_video_with_enhanced_tracking(
    video_path=VIDEO_PATH,
    output_path=OUTPUT_VIDEO_PATH,
    model_path=MODEL_PATH,
    buffer_size=10,
    speed_threshold=500.0,
    confidence_threshold=0.3)

# def callback(frame):
    global frame_number
    frame_number += 1
    
    # YOLO detection
    result = model(frame)[0]
    detections = sv.Detections.from_ultralytics(result)
    
    # Filter detections by borders and class if needed
    detections = detections[
        (detections.xyxy[:, 0] > border_x) & 
        (detections.xyxy[:, 1] > border_y) & 
        (detections.xyxy[:, 2] < width - border_x) & 
        (detections.xyxy[:, 3] < height - border_y)
    ]
    
    # Optional: Filter by specific class (uncomment if needed)
    # detections = detections[(detections.class_id == 32)]  # Replace 32 with your ball class ID
    
    # Enhanced tracking with outlier removal
    tracked_detections = tracker.update(detections, frame_number)
    
    # Advanced visualization
    annotated_frame = visualizer.annotate_frame(
        frame, tracked_detections, tracker, frame_number
    )
    
    all_detections.append(tracked_detections)
    
    # Optional: Print tracking info every 100 frames
    if frame_number % 100 == 0:
        stats = tracker.get_stats()
        print(f"Frame {frame_number}: {stats}")
    
    return annotated_frame

# Process video
# sv.process_video(
#     source_path=VIDEO_PATH, 
#     target_path=OUTPUT_VIDEO_PATH, 
#     callback=callback
# )

# Print final statistics
# print("\n=== Final Tracking Statistics ===")
# final_stats = tracker.get_stats()
# for key, value in final_stats.items():
#     print(f"{key}: {value}")

# # Get trajectory and speeds for analysis
# trajectory = tracker.get_trajectory()
# speeds = tracker.get_speeds()

# print(f"\nTrajectory points: {len(trajectory)}")
# print(f"Average speed: {np.mean(speeds):.2f} pixels/frame" if speeds else "No speed data")
# print(f"Max speed: {np.max(speeds):.2f} pixels/frame" if speeds else "No speed data")



IndentationError: unexpected indent (3494798332.py, line 522)

In [ ]:
# Weighted averaging giving prevence to recent ones.
import supervision as sv
import numpy as np
from collections import deque

"""Exponential Decay Weights: Recent frames get exponentially higher weights using a decay_factor parameter (default 0.9)
Normalized Weights: Weights sum to 1.0, ensuring the weighted average stays within reasonable bounds
Weighted Centroid Calculation: Instead of simple mean, uses np.sum(weights * positions)
Configurable Decay: You can adjust how much recent vs. historical data matters via decay_factor"""
class BallTracker:
    def __init__(self, buffer_size: int = 10, decay_factor: float = 0.9):
        """
        Initialize the ball tracker with weighted averaging.
        
        Args:
            buffer_size: Maximum number of frames to keep in buffer
            decay_factor: Weight decay factor (0 < decay_factor < 1). 
                         Higher values give more weight to recent frames.
        """
        self.buffer = deque(maxlen=buffer_size)
        self.decay_factor = decay_factor
        self.buffer_size = buffer_size

    def _calculate_weights(self, buffer_length: int) -> np.ndarray:
        """
        Calculate exponential decay weights for the buffer.
        Most recent frame gets weight 1.0, earlier frames get progressively smaller weights.
        """
        if buffer_length == 0:
            return np.array([])
        
        # Create weights: [decay^(n-1), decay^(n-2), ..., decay^1, decay^0]
        # where n is buffer_length and decay^0 = 1.0 for the most recent frame
        weights = np.array([self.decay_factor ** (buffer_length - 1 - i) 
                           for i in range(buffer_length)])
        
        # Normalize weights to sum to 1
        return weights / np.sum(weights)

    def _calculate_weighted_centroid(self) -> np.ndarray:
        """
        Calculate weighted centroid from buffer using exponential decay weights.
        """
        if len(self.buffer) == 0:
            return np.array([0, 0])
        
        # Get weights for current buffer
        weights = self._calculate_weights(len(self.buffer))
        
        # Calculate weighted average for each frame's detections
        weighted_positions = []
        
        for i, positions in enumerate(self.buffer):
            if len(positions) > 0:
                # If multiple detections in a frame, take their mean
                frame_centroid = np.mean(positions, axis=0)
                weighted_positions.append(weights[i] * frame_centroid)
        
        if len(weighted_positions) == 0:
            return np.array([0, 0])
        
        # Sum all weighted positions to get final weighted centroid
        return np.sum(weighted_positions, axis=0)

    def update(self, detections: sv.Detections) -> sv.Detections:
        """
        Update tracker with new detections and return the best detection.
        """
        # Get center coordinates of current detections
        xy = detections.get_anchors_coordinates(sv.Position.CENTER)
        
        # Add current detections to buffer
        self.buffer.append(xy)
        
        # If no detections, return empty
        if len(detections) == 0:
            return detections
        
        # Calculate weighted centroid from historical data
        weighted_centroid = self._calculate_weighted_centroid()
        
        # Find detection closest to weighted centroid
        distances = np.linalg.norm(xy - weighted_centroid, axis=1)
        index = np.argmin(distances)
        
        return detections[[index]]

    def get_predicted_position(self) -> np.ndarray:
        """
        Get the current predicted position (weighted centroid) without new detections.
        Useful for visualization or when no detections are found.
        """
        return self._calculate_weighted_centroid()
    
    def reset(self):
        """Reset the tracker buffer."""
        self.buffer.clear()

In [ ]:

class PhysicsBasedFilter:
    def __init__(self, max_velocity_change=50, max_acceleration=20):
        self.position_history = deque(maxlen=3)
        self.max_velocity_change = max_velocity_change
        self.max_acceleration = max_acceleration
    
    def is_motion_consistent(self, new_position):
        if len(self.position_history) < 2:
            return True
        
        # Calculate current and previous velocities
        prev_pos = self.position_history[-1]
        prev_prev_pos = self.position_history[-2]
        
        prev_velocity = prev_pos - prev_prev_pos
        current_velocity = new_position - prev_pos
        
        # Check velocity change (acceleration)
        velocity_change = np.linalg.norm(current_velocity - prev_velocity)
        
        return velocity_change < self.max_velocity_change
 


In [ ]:
class SizeConsistentTracker:
    def __init__(self, size_tolerance=0.3):
        self.size_history = deque(maxlen=10)
        self.size_tolerance = size_tolerance
    
    def filter_by_size(self, detections):
        if len(self.size_history) == 0:
            return detections
        
        expected_area = np.mean(self.size_history)
        areas = (detections.xyxy[:, 2] - detections.xyxy[:, 0]) * \
                (detections.xyxy[:, 3] - detections.xyxy[:, 1])
        
        size_ratios = areas / expected_area
        valid_mask = (size_ratios > (1 - self.size_tolerance)) & \
                    (size_ratios < (1 + self.size_tolerance))
        
        return detections[valid_mask]

In [ ]:
# Dynamic thresholding based on detection quality
def filter_by_confidence(detections, min_confidence=0.5, adaptive=True):
    if adaptive and len(detections) > 1:
        # Use mean + std to set adaptive threshold
        scores = detections.confidence
        threshold = np.mean(scores) - 0.5 * np.std(scores)
        threshold = max(threshold, min_confidence)
    else:
        threshold = min_confidence
    
    return detections[detections.confidence > threshold]

In [ ]:
class TemporalConsistencyFilter:
    def __init__(self, min_track_length=3, max_gap=2):
        self.track_states = {}  # track_id -> consecutive_frames
        self.gap_counts = {}    # track_id -> gap_count
        self.min_track_length = min_track_length
        self.max_gap = max_gap
    
    def filter_tracks(self, detections, track_ids):
        valid_indices = []
        
        for i, track_id in enumerate(track_ids):
            # Update track state
            if track_id in self.track_states:
                self.track_states[track_id] += 1
                self.gap_counts[track_id] = 0
            else:
                self.track_states[track_id] = 1
                self.gap_counts[track_id] = 0
            
            # Keep track if it's been consistent long enough
            if self.track_states[track_id] >= self.min_track_length:
                valid_indices.append(i)
        
        return detections[valid_indices]

In [ ]:
class ContextualFilter:
    def __init__(self, field_boundaries=None, player_positions=None):
        self.field_boundaries = field_boundaries
        self.player_positions = player_positions
    
    def filter_by_context(self, detections):
        positions = detections.get_anchors_coordinates(sv.Position.CENTER)
        valid_mask = np.ones(len(detections), dtype=bool)
        
        # Remove detections outside field boundaries
        if self.field_boundaries:
            x_min, y_min, x_max, y_max = self.field_boundaries
            valid_mask &= (positions[:, 0] >= x_min) & (positions[:, 0] <= x_max)
            valid_mask &= (positions[:, 1] >= y_min) & (positions[:, 1] <= y_max)
        
        # Remove detections too close to players (likely false positives)
        if self.player_positions is not None:
            for player_pos in self.player_positions:
                distances = np.linalg.norm(positions - player_pos, axis=1)
                valid_mask &= distances > 30  # Minimum distance from players
        
        return detections[valid_mask]

In [ ]:
#Train a model maybe LSTM to identify real balls from false positives: based on features such as area, confidence, and making other features. 

In [ ]:
#smoothning based on tracking Ids, built in supervision.
import supervision as sv

from ultralytics import YOLO

video_info = sv.VideoInfo.from_video_path(video_path=<SOURCE_FILE_PATH>)
frame_generator = sv.get_video_frames_generator(source_path=<SOURCE_FILE_PATH>)

model = YOLO(<MODEL_PATH>)
tracker = sv.ByteTrack(frame_rate=video_info.fps)
smoother = sv.DetectionsSmoother()

box_annotator = sv.BoxAnnotator()

with sv.VideoSink(<TARGET_FILE_PATH>, video_info=video_info) as sink:
    for frame in frame_generator:
        result = model(frame)[0]
        detections = sv.Detections.from_ultralytics(result)
        detections = tracker.update_with_detections(detections)
        detections = smoother.update_with_detections(detections)

        annotated_frame = box_annotator.annotate(frame.copy(), detections)
        sink.write_frame(annotated_frame)

In [ ]:
# import supervision as sv
# import numpy as np
# from ultralytics import YOLO
# from collections import deque
# from typing import Optional, List, Tuple, Dict, Any
# import cv2

# class EnhancedBallTracker:
#     def __init__(
#         self, 
#         buffer_size: int = 10,
#         speed_threshold: float = 500.0,  # pixels per frame
#         confidence_threshold: float = 0.5,
#         max_distance_threshold: float = 200.0,  # max pixels from centroid
#         min_detections_for_speed: int = 2
#     ):
#         self.buffer = deque(maxlen=buffer_size)
#         self.position_buffer = deque(maxlen=buffer_size)
#         self.frame_numbers = deque(maxlen=buffer_size)
#         self.confidence_buffer = deque(maxlen=buffer_size)
#         self.speed_buffer = deque(maxlen=buffer_size - 1)
        
#         # Thresholds
#         self.speed_threshold = speed_threshold
#         self.confidence_threshold = confidence_threshold
#         self.max_distance_threshold = max_distance_threshold
#         self.min_detections_for_speed = min_detections_for_speed
        
#         # Frame tracking
#         self.current_frame = 0
        
#         # Statistics
#         self.stats = {
#             'total_detections': 0,
#             'outliers_removed': 0,
#             'speed_outliers': 0,
#             'distance_outliers': 0,
#             'confidence_outliers': 0
#         }

#     def calculate_speed(self, pos1: np.ndarray, pos2: np.ndarray, frame_diff: int = 1) -> float:
#         """Calculate speed between two positions in pixels per frame"""
#         if frame_diff <= 0:
#             return 0.0
#         distance = np.linalg.norm(pos2 - pos1)
#         return distance / frame_diff

#     def is_speed_outlier(self, current_pos: np.ndarray) -> bool:
#         """Check if current position creates unrealistic speed"""
#         if len(self.position_buffer) < self.min_detections_for_speed:
#             return False
        
#         prev_pos = self.position_buffer[-1]
#         prev_frame = self.frame_numbers[-1]
#         frame_diff = self.current_frame - prev_frame
        
#         if frame_diff <= 0:
#             return False
            
#         speed = self.calculate_speed(prev_pos, current_pos, frame_diff)
#         return speed > self.speed_threshold

#     def is_distance_outlier(self, current_pos: np.ndarray) -> bool:
#         """Check if current position is too far from trajectory centroid"""
#         if len(self.position_buffer) < 2:
#             return False
        
#         # Calculate centroid of recent positions
#         recent_positions = np.array(list(self.position_buffer))
#         centroid = np.mean(recent_positions, axis=0)
#         distance = np.linalg.norm(current_pos - centroid)
        
#         return distance > self.max_distance_threshold

#     def filter_detections(self, detections: sv.Detections) -> sv.Detections:
#         """Filter detections based on confidence and other criteria"""
#         if len(detections) == 0:
#             return detections
        
#         # Filter by confidence
#         high_conf_mask = detections.confidence >= self.confidence_threshold
        
#         if not np.any(high_conf_mask):
#             # If no high confidence detections, keep the highest confidence one
#             best_idx = np.argmax(detections.confidence)
#             return detections[[best_idx]]
        
#         return detections[high_conf_mask]

#     def select_best_detection(self, detections: sv.Detections) -> Optional[sv.Detections]:
#         """Select the best detection from multiple candidates"""
#         if len(detections) == 0:
#             return None
        
#         if len(detections) == 1:
#             return detections
        
#         xy = detections.get_anchors_coordinates(sv.Position.CENTER)
        
#         # If we have trajectory history, use it to select best detection
#         if len(self.position_buffer) > 0:
#             centroid = np.mean(np.array(list(self.position_buffer)), axis=0)
#             distances = np.linalg.norm(xy - centroid, axis=1)
#             best_idx = np.argmin(distances)
#             return detections[[best_idx]]
        
#         # Otherwise, select highest confidence
#         best_idx = np.argmax(detections.confidence)
#         return detections[[best_idx]]

#     def update_buffers(self, detection: sv.Detections):
#         """Update all tracking buffers with new detection"""
#         if len(detection) == 0:
#             return
        
#         center = detection.get_anchors_coordinates(sv.Position.CENTER)[0]
#         confidence = detection.confidence[0]
        
#         # Update buffers
#         self.buffer.append(detection.get_anchors_coordinates(sv.Position.CENTER))
#         self.position_buffer.append(center)
#         self.frame_numbers.append(self.current_frame)
#         self.confidence_buffer.append(confidence)
        
#         # Calculate and store speed if possible
#         if len(self.position_buffer) >= 2:
#             prev_pos = list(self.position_buffer)[-2]
#             prev_frame = list(self.frame_numbers)[-2]
#             frame_diff = self.current_frame - prev_frame
#             speed = self.calculate_speed(prev_pos, center, frame_diff)
#             self.speed_buffer.append(speed)

#     def update(self, detections: sv.Detections, frame_number: Optional[int] = None) -> sv.Detections:
#         """Main update method with outlier removal"""
#         if frame_number is not None:
#             self.current_frame = frame_number
#         else:
#             self.current_frame += 1
        
#         self.stats['total_detections'] += len(detections)
        
#         if len(detections) == 0:
#             return detections
        
#         # Step 1: Filter by confidence
#         filtered_detections = self.filter_detections(detections)
        
#         # Step 2: Check for outliers
#         valid_detections = []
        
#         for i in range(len(filtered_detections)):
#             single_detection = filtered_detections[[i]]
#             center = single_detection.get_anchors_coordinates(sv.Position.CENTER)[0]
#             confidence = single_detection.confidence[0]
            
#             is_outlier = False
            
#             # Check confidence outlier
#             if confidence < self.confidence_threshold:
#                 is_outlier = True
#                 self.stats['confidence_outliers'] += 1
            
#             # Check speed outlier
#             elif self.is_speed_outlier(center):
#                 is_outlier = True
#                 self.stats['speed_outliers'] += 1
            
#             # Check distance outlier
#             elif self.is_distance_outlier(center):
#                 is_outlier = True
#                 self.stats['distance_outliers'] += 1
            
#             if not is_outlier:
#                 valid_detections.append(i)
        
#         # Step 3: Select best detection from valid ones
#         if valid_detections:
#             valid_detections_obj = filtered_detections[valid_detections]
#             best_detection = self.select_best_detection(valid_detections_obj)
#         else:
#             # If all are outliers, skip this frame or use fallback logic
#             self.stats['outliers_removed'] += len(filtered_detections)
#             return sv.Detections.empty()
        
#         # Step 4: Update buffers
#         if best_detection and len(best_detection) > 0:
#             self.update_buffers(best_detection)
#             return best_detection
        
#         return sv.Detections.empty()

#     def get_trajectory(self) -> List[Tuple[float, float]]:
#         """Get the current trajectory as list of (x, y) coordinates"""
#         return [(pos[0], pos[1]) for pos in self.position_buffer]

#     def get_speeds(self) -> List[float]:
#         """Get the speed history"""
#         return list(self.speed_buffer)

#     def get_stats(self) -> Dict[str, Any]:
#         """Get tracking statistics"""
#         stats = self.stats.copy()
#         if stats['total_detections'] > 0:
#             stats['outlier_percentage'] = (stats['outliers_removed'] / stats['total_detections']) * 100
#         else:
#             stats['outlier_percentage'] = 0
#         return stats

#     def reset(self):
#         """Reset the tracker"""
#         self.buffer.clear()
#         self.position_buffer.clear()
#         self.frame_numbers.clear()
#         self.confidence_buffer.clear()
#         self.speed_buffer.clear()
#         self.current_frame = 0
#         self.stats = {
#             'total_detections': 0,
#             'outliers_removed': 0,
#             'speed_outliers': 0,
#             'distance_outliers': 0,
#             'confidence_outliers': 0
#         }


# class VersatileVisualizer:
#     """Versatile visualization class for ball tracking results"""
    
#     def __init__(
#         self,
#         show_trajectory: bool = True,
#         show_speed: bool = True,
#         show_confidence: bool = True,
#         trajectory_length: int = 30,
#         trajectory_thickness: int = 3
#     ):
#         self.show_trajectory = show_trajectory
#         self.show_speed = show_speed
#         self.show_confidence = show_confidence
#         self.trajectory_length = trajectory_length
#         self.trajectory_thickness = trajectory_thickness
        
#         # Annotation components
#         self.box_annotator = sv.BoxAnnotator(
#             color=sv.Color.GREEN,
#             thickness=2
#         )
#         self.label_annotator = sv.LabelAnnotator(
#             color=sv.Color.GREEN,
#             text_color=sv.Color.WHITE,
#             text_scale=0.7,
#             text_thickness=2
#         )
#         self.trace_annotator = sv.TraceAnnotator(
#             color=sv.Color.RED,
#             thickness=self.trajectory_thickness,
#             trace_length=self.trajectory_length
#         )
        
#         # Custom colors for different confidence levels
#         self.confidence_colors = {
#             'high': sv.Color.GREEN,    # >= 0.8
#             'medium': sv.Color.YELLOW, # 0.5-0.8
#             'low': sv.Color.RED        # < 0.5
#         }

#     def get_confidence_color(self, confidence: float) -> sv.Color:
#         """Get color based on confidence level"""
#         if confidence >= 0.8:
#             return self.confidence_colors['high']
#         elif confidence >= 0.5:
#             return self.confidence_colors['medium']
#         else:
#             return self.confidence_colors['low']

#     def create_labels(
#         self, 
#         detections: sv.Detections, 
#         tracker: EnhancedBallTracker,
#         frame_number: int
#     ) -> List[str]:
#         """Create informative labels for detections"""
#         labels = []
        
#         for i, (class_id, confidence) in enumerate(zip(detections.class_id, detections.confidence)):
#             label_parts = []
            
#             # Basic info
#             label_parts.append(f"Ball: {confidence:.2f}")
            
#             # Speed info
#             if self.show_speed and len(tracker.get_speeds()) > 0:
#                 current_speed = tracker.get_speeds()[-1]
#                 label_parts.append(f"Speed: {current_speed:.1f}")
            
#             # Frame number
#             label_parts.append(f"Frame: {frame_number}")
            
#             labels.append(" | ".join(label_parts))
        
#         return labels

#     def draw_trajectory_info(
#         self, 
#         frame: np.ndarray, 
#         tracker: EnhancedBallTracker
#     ) -> np.ndarray:
#         """Draw additional trajectory information on frame"""
#         if not self.show_trajectory:
#             return frame
        
#         trajectory = tracker.get_trajectory()
#         if len(trajectory) < 2:
#             return frame
        
#         # Draw trajectory line
#         points = np.array(trajectory, dtype=np.int32)
        
#         for i in range(1, len(points)):
#             # Fade color based on age
#             alpha = i / len(points)
#             color = (0, int(255 * alpha), 0)  # Green with fading
#             cv2.line(frame, tuple(points[i-1]), tuple(points[i]), color, 2)
        
#         # Draw speed graph (mini visualization)
#         if self.show_speed and len(tracker.get_speeds()) > 1:
#             self.draw_speed_graph(frame, tracker.get_speeds())
        
#         return frame

#     def draw_speed_graph(self, frame: np.ndarray, speeds: List[float]) -> None:
#         """Draw a mini speed graph on the frame"""
#         if len(speeds) < 2:
#             return
        
#         # Position for mini graph (top-right corner)
#         graph_w, graph_h = 200, 100
#         start_x = frame.shape[1] - graph_w - 20
#         start_y = 20
        
#         # Background
#         cv2.rectangle(frame, (start_x, start_y), (start_x + graph_w, start_y + graph_h), (0, 0, 0), -1)
#         cv2.rectangle(frame, (start_x, start_y), (start_x + graph_w, start_y + graph_h), (255, 255, 255), 2)
        
#         # Scale speeds to graph height
#         max_speed = max(speeds) if speeds else 1
#         scaled_speeds = [(speed / max_speed) * (graph_h - 20) for speed in speeds[-20:]]  # Last 20 speeds
        
#         # Draw speed line
#         points = []
#         for i, speed in enumerate(scaled_speeds):
#             x = start_x + 10 + int((i / len(scaled_speeds)) * (graph_w - 20))
#             y = start_y + graph_h - 10 - int(speed)
#             points.append((x, y))
        
#         if len(points) > 1:
#             for i in range(1, len(points)):
#                 cv2.line(frame, points[i-1], points[i], (0, 255, 0), 2)
        
#         # Add labels
#         cv2.putText(frame, f"Speed: {speeds[-1]:.1f}", (start_x + 5, start_y + 15), 
#                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

#     def annotate_frame(
#         self, 
#         frame: np.ndarray, 
#         detections: sv.Detections, 
#         tracker: EnhancedBallTracker,
#         frame_number: int
#     ) -> np.ndarray:
#         """Main annotation method"""
#         annotated_frame = frame.copy()
        
#         if len(detections) == 0:
#             return self.draw_trajectory_info(annotated_frame, tracker)
        
#         # Create dynamic box annotator based on confidence
#         if self.show_confidence and len(detections.confidence) > 0:
#             confidence = detections.confidence[0]
#             color = self.get_confidence_color(confidence)
#             dynamic_box_annotator = sv.BoxAnnotator(color=color, thickness=2)
#             dynamic_label_annotator = sv.LabelAnnotator(color=color, text_color=sv.Color.WHITE)
#         else:
#             dynamic_box_annotator = self.box_annotator
#             dynamic_label_annotator = self.label_annotator
        
#         # Annotate with boxes and labels
#         annotated_frame = dynamic_box_annotator.annotate(annotated_frame, detections)
        
#         labels = self.create_labels(detections, tracker, frame_number)
#         annotated_frame = dynamic_label_annotator.annotate(annotated_frame, detections, labels=labels)
        
#         # Add trajectory visualization
#         annotated_frame = self.draw_trajectory_info(annotated_frame, tracker)
        
#         return annotated_frame


# # Usage Example
# def process_video_with_enhanced_tracking(
#     video_path: str,
#     output_path: str,
#     model_path: str,
#     **tracker_kwargs
# ):
#     """Process video with enhanced ball tracking"""
    
#     # Initialize components
#     model = YOLO(model_path)
#     tracker = EnhancedBallTracker(**tracker_kwargs)
#     visualizer = VersatileVisualizer()
    
#     # Video info
#     video_info = sv.VideoInfo.from_video_path(video_path)
#     width, height = video_info.resolution_wh
    
#     # Border filtering
#     percentage_to_exclude_from_sides = 0.01
#     border_x = width * percentage_to_exclude_from_sides
#     border_y = height * percentage_to_exclude_from_sides
    
#     all_detections = []
#     frame_number = 0
    
#     def callback(frame: np.ndarray, index) -> np.ndarray:
#         nonlocal frame_number
#         frame_number += 1
        
#         # Run YOLO detection
#         result = model(frame)[0]
#         detections = sv.Detections.from_ultralytics(result)
        
#         # Filter detections (borders, class, etc.)
#         detections = detections[
#             (detections.xyxy[:, 0] > border_x) & 
#             (detections.xyxy[:, 1] > border_y) & 
#             (detections.xyxy[:, 2] < width - border_x) & 
#             (detections.xyxy[:, 3] < height - border_y)
#         ]
        
#         # Keep only highest confidence detection initially
#         if len(detections) > 1:
#             sort_keep_indices = np.argsort(detections.confidence)[::-1][:1]
#             detections = detections[sort_keep_indices]
        
#         # Update tracker with outlier removal
#         tracked_detections = tracker.update(detections, frame_number)
        
#         # Visualize
#         annotated_frame = visualizer.annotate_frame(
#             frame, tracked_detections, tracker, frame_number
#         )
        
#         all_detections.append(tracked_detections)
        
#         return annotated_frame
    
#     # Process video
#     sv.process_video(
#         source_path=video_path,
#         target_path=output_path,
#         callback=callback
#     )
    
#     # Print statistics
#     stats = tracker.get_stats()
#     print("\n=== Tracking Statistics ===")
#     for key, value in stats.items():
#         print(f"{key}: {value}")
    
#     return all_detections, tracker

# import supervision as sv
# import numpy as np
# from ultralytics import YOLO

# # Your file paths
# VIDEO_PATH = '/Users/spectatr/Downloads/nsl_match2/683efcc10388617b7ff4b9cf/input_001a21f5-580a-4280-98f2-49d4e3fbd28d1748958688.mp4'
# OUTPUT_VIDEO_PATH = '/Users/spectatr/Downloads/testing2.mp4'
# MODEL_PATH = '/Users/spectatr/Downloads/best_datav10_rectified_nano_1920_e261_final.pt'

# # Initialize enhanced tracker with custom parameters
# tracker = EnhancedBallTracker(
#     buffer_size=10,
#     speed_threshold=500.0,  # Adjust based on your sport (pixels per frame)
#     confidence_threshold=0.3,  # Lower threshold for initial filtering
#     max_distance_threshold=200.0,  # Max distance from trajectory centroid
#     min_detections_for_speed=2
# )

# # Initialize versatile visualizer
# visualizer = VersatileVisualizer(
#     show_trajectory=True,
#     show_speed=True,
#     show_confidence=True,
#     trajectory_length=30
# )

# # Alternative: Use the convenience function
# all_detections, tracker = process_video_with_enhanced_tracking(
#     video_path=VIDEO_PATH,
#     output_path=OUTPUT_VIDEO_PATH,
#     model_path=MODEL_PATH,
#     buffer_size=10,
#     speed_threshold=500.0,
#     confidence_threshold=0.3
# )